---
# <div align="center"><font color='blue'> COSC 2779/2972 | Deep Learning </font></div>
## <div align="center"><font color='blue'> Assignment 1: Introduction to Deep Convolutional Neural Networks</font></div>
### <div align="center">by Daniil Pkhakadze (s4101606) RMIT semester 2 2026</div>
---

## The problem.
A drone company (RedKite Aerospace) needs its drones to be able to read hand signals from the person receiving the drone delivery. Different gestures need to map to different commands and the drone needs to be able to differentiate
between them. A segmentor that already exists on the aircraft will cut the relevant parts out (the gestures) and pass each one to the model as 5 frames.
<br>
The task at hand is a **13-class classification problem** over these pre-segmented frames: 5 frames go into the model, and it outputs one label for the given gesture.
<br>
The 13 classes (gestures) that exist in the dataset are:
- `AllClear`
- `HaveCommand`
- `Hover`
- `Land`
- `LandingDirection`
- `MoveAhead`
- `MoveDownward`
- `MoveToLeft`
- `MoveToRight`
- `MoveUpward`
- `NotClear`
- `SlowDown`
- `WaveOff`
<br>
I will be building a classifier model. There are some restrictions: the computer is already overloaded as it is running navigation, mapping, comms etc., so there is very little space and power to run a complicated classifier model. Due to that, we are restricted to only use a 2-D convolutional backbone. The model also has to run inference on one instance at a time on an embedded board, which essentially means that it needs to stay small enough for real-time operation to be possible.
<br>
A 2-D CNN by itself would not be able to achieve satisfactory accuracy results on such a small dataset. Using an existing ImageNet-pretrained backbone could be a solution - that way the model doesn't have to learn all the visual features from scratch. We can also apply data augmentation, which would help us get more use out of the small dataset that we have.
<br>
Another potential architectural issue is that a 2-D CNN is only able to take 1 image as input to produce 1 output label, so normally it doesn't have a way to use the sequence of 5 frames that we have.
Getting the 5 frames into a 2-D CNN is an issue that will need to be addressed by attempting various appraoches to this problem.

1. **Singular frame** - classify simple based on one frame out of 5, which is the simplest possible approach. This will serve as the *baseline*.
2. **Early frame fusion** - we will stack the 5 frames into one as channels - this is a workaround for the network to get all 5 images in one input.
3. **Late frame fusion** - we will run the same backbone on each of the five frames in the sequence and then we will combine the resulting features in the head.
<br>
The given dataset itself could prove to be a problem - it was recorded with RedKite's own staff. On the actual day of the demo, a member of the client's staff will actually be doing the gestures for testing, which will be a person that the model has never seen before in an unseen location.

## Setup.
First, we need to import all the core libraries we will need for development. We will also be setting a fixed random seed to make sure our results are always reproducible.

In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Step 1. Load the dataset.
We will define a "Dataset" class for PyTorch to load our image data. Each training example is a folder containing 5 frames:
- `__getitem__` loads the 5 frames, then applies the tranform to each one and then puts them into one tensor.
Source is the Week 4 Lab.

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class GestureDataset(Dataset):
    def __init__(self, instances, transform=None):
        self.transform = transform
        self.instances = instances
        self.targets = [c for _, _, c in instances]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        inst_dir, _, label = self.instances[idx]
        frame_files = sorted(f for f in os.listdir(inst_dir) if f.endswith(('.jpg', '.jpeg', '.png')))
        frames = []
        for fname in frame_files:
            img = Image.open(os.path.join(inst_dir, fname)).convert('RGB')
            if self.transform is not None:
                img = self.transform(img)
            frames.append(img)
        frames = torch.stack(frames)
        return frames, label

First, let's setup the data folder and create a list of all the 13 gesture classes. Also we will create a dictionary that maps each class name to an integer index for simplicity.

In [7]:
DATA_FOLDER = 'data_resized'

CLASS_NAMES = ['AllClear', 'HaveCommand', 'Hover', 'Land', 'LandingDirection',
               'MoveAhead', 'MoveDownward', 'MoveToLeft', 'MoveToRight',
               'MoveUpward', 'NotClear', 'SlowDown', 'WaveOff']
NUM_CLASSES = len(CLASS_NAMES)

class_to_index = {}
for i in range (NUM_CLASSES):
    class_to_index[CLASS_NAMES[i]] = i

We will now inspect the folder structure. For each out of the 13 index classes and each subject (S) within, every `InstanceN` folder will be a singular training example (since it contains a single gesture with 5 frames). We need to store information, such as the path, subject number and the class label.
This is based on code from Week 6 lab, except data from this dataset goes one folder deeper since it is organized like so: *class -> subject -> instance*.

In [13]:
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_FOLDER, cls)
    print(f'{cls}:')
    for subj in sorted(os.listdir(cls_dir)):
        subj_dir = os.path.join(cls_dir, subj)
        n = 0
        if os.path.isdir(subj_dir):
            for inst in os.listdir(subj_dir):
                if os.path.isdir(os.path.join(subj_dir, inst)):
                    n+=1
            print(f'  {subj}: {n}')
    print()

AllClear:
  S1: 8
  S11: 9
  S12: 8
  S13: 4
  S14: 10
  S15: 7
  S3: 9
  S4: 7
  S5: 6
  S6: 9
  S9: 10

HaveCommand:
  S1: 9
  S11: 7
  S12: 7
  S13: 9
  S14: 8
  S15: 9
  S3: 10
  S4: 7
  S5: 5
  S6: 5
  S9: 9

Hover:
  S1: 9
  S11: 9
  S12: 11
  S13: 11
  S4: 13
  S5: 16
  S6: 13

Land:
  S1: 17
  S11: 19
  S12: 18
  S13: 18
  S4: 17
  S5: 17
  S6: 16

LandingDirection:
  S1: 7
  S11: 9
  S12: 8
  S13: 8
  S4: 7
  S5: 4
  S6: 5

MoveAhead:
  S1: 10
  S11: 8
  S12: 10
  S13: 9
  S14: 8
  S15: 10
  S3: 10
  S4: 6
  S5: 8
  S6: 10
  S9: 10

MoveDownward:
  S1: 8
  S11: 5
  S12: 8
  S13: 5
  S4: 7
  S5: 6
  S6: 7

MoveToLeft:
  S1: 9
  S11: 9
  S12: 8
  S13: 9
  S14: 9
  S15: 9
  S3: 10
  S4: 7
  S5: 7
  S6: 8
  S9: 10

MoveToRight:
  S1: 9
  S11: 8
  S12: 9
  S13: 8
  S14: 8
  S15: 9
  S3: 9
  S4: 10
  S5: 6
  S6: 9
  S9: 10

MoveUpward:
  S1: 10
  S11: 9
  S12: 9
  S13: 10
  S4: 8
  S5: 7
  S6: 9

NotClear:
  S1: 5
  S11: 9
  S12: 7
  S13: 9
  S14: 10
  S15: 9
  S3: 10
  S4: 6
  S5: 

As we can see, the subject IDs are S1 to S15, however only 11 of them are actually present in the dataset - S2, S7, S8, S10 are the ones NOT present. In the context of this dataset a subject is a unique person.
AWe can also see that not each one of the 11 subjects has performed every single one of the 13 gestures:
- 7 gestures done by all 11 people
- 6 gestures were however only done by 7 people (`Hover`, `Land`, `LandingDirection`, `MoveDownward`, `MoveUpward`, `WaveOff`). 4 people that did NOT do the gesture are S3, S9, S14, S15.